Post-processing of CFD solution
- by py vista library

In [7]:
from scipy.io import mmwrite, mmread
import pyvista as pv
import numpy as np
import pandas as pd
from pyau3d.utils import PltFileUtils, UnkFileUtils
from pyau3d.pv.loader.au3d import arrays2vtk

In [8]:
def specific_energy(rho, p, ux, uy, uz, gamma=1.4):
    return p / ((gamma - 1.0) * rho) + 0.5 * (ux**2 + uy**2 + uz**2)

def conservative_variables(rst, GAMMA):
    """Return conservative variables U on the surface."""
    E = specific_energy(rst.rho, rst.p, rst.ux, rst.uy, rst.uz, GAMMA)

    U = np.column_stack((
        rst.rho,
        rst.rho * rst.ux,
        rst.rho * rst.uy,
        rst.rho * rst.uz,
        rst.rho * E
    ))
    return U

In [9]:
# 1. Read neccessary files
Mesh = 3060
Re   = 60
Mach = 0.2
gamma = 1.4
R_gas = 287.0          # confirm units match the solver


dir = f"C:/Users/User/Git/flux_jacobian/cases/cylinder_{Mesh}_Re{Re}_M{Mach}"
pltfile = PltFileUtils(f"{dir}/cylinder.plt")
rstfile = UnkFileUtils(f"{dir}/cylinder.rst", extend=False)

fortfile = pd.read_csv(f"{dir}/fort.864", sep=r'\s+', header=None).to_numpy()

rstfile._primitive()
U_list = conservative_variables(rstfile, gamma)
coord  = pltfile.coord

In [10]:
# transform PLT to VTK
mesh = arrays2vtk(pltfile)

# extract the grid from the pv object
domain = pv.wrap(mesh.GetBlock(0))    # "Domain 1"  →  pv.MultiBlock
block  = pv.wrap(domain.GetBlock(0))  # "Volume"    →  pv.UnstructuredGrid

# overlay U_list on the VTK file 
names = ['rho', 'rhou', 'rhov', 'rhow', 'rhoE']

for i in range(5):
    block.point_data[names[i]] = U_list[:,i]

In [11]:
# plot mesh

pl = pv.Plotter()
pl.add_mesh(block, style='wireframe', color='white', line_width=0.5)
# pl.view_xy()
pl.show()

Widget(value='<iframe src="http://localhost:54155/index.html?ui=P_0x21316fe17f0_0&reconnect=auto" class="pyvis…

Node 1151  x=-0.0350  y=0.6814  err=1.1474e-03


In [13]:
# ── Plot one field  ───────────────────────────────────────────────────────────

# for i in range(5):
#     pl = pv.Plotter(notebook = False)
#     pl.add_mesh(block, scalars=names[i], cmap='RdBu', show_edges=False, edge_color='grey', opacity=0.99)
#     # pl.add_scalar_bar(title='dF_rho / d(rho*u)')
#     pl.view_xy()
#     pl.show()
#     pl.save_graphic(f"Re{Re}_M{Mach}_{names[i]}.svg")


def print_pick(point):
    # finds nearest point and prints its value
    pid = block.find_closest_point(point)
    err = block.point_data['rhou'][pid]
    xy  = block.points[pid, :2]
    print(f"Node {pid}  x={xy[0]:.4f}  y={xy[1]:.4f}  err={err:.4e}")

pl = pv.Plotter(notebook = False)
pl.add_mesh(block, scalars="rhou", cmap='RdBu', show_edges=True, edge_color='grey', opacity=0.9, pickable = True)
pl.enable_point_picking(callback=print_pick, show_message=True,
                        font_size=10, color='black', point_size=10)
# pl.add_scalar_bar(title='dF_rho / d(rho*u)')
pl.view_xy()
pl.show()



Node 2654  x=0.0250  y=0.4994  err=0.0000e+00
Node 976  x=0.0342  y=0.6815  err=1.0671e-03
